# 02 - Fine-Tuning IndoT5 dengan Grid Search Hyperparameter

Total eksperimen: **1 baseline + 18 grid search = 19 run**

| # | Konfigurasi | Epoch | Batch Size | Learning Rate |
|---|---|---|---|---|
| 0 | **Baseline** (default, tanpa tuning) | - | - | - |
| 1–18 | **Grid Search** | 5, 10, 15 | 4, 8 | 1e-4, 3e-5, 5e-5 |

> Setiap run dievaluasi dengan ROUGE pada `test.csv`. Model dengan ROUGE-L tertinggi disimpan sebagai model final.

In [1]:
from huggingface_hub import login
from google.colab import userdata
import os

try:
    # Ambil token "HF_TOKEN" langsung dari Secret Colab
    my_token = userdata.get('HF_TOKEN')

    if my_token:
        # Login menggunakan token
        login(token=my_token, add_to_git_credential=True)
        print("Berhasil login ke Hugging Face menggunakan Secret Colab!")
    else:
        print("Token HF_TOKEN tidak ditemukan di Secrets Colab.")

except Exception as e:
    print(f"Gagal mengambil secret: {e}")

Berhasil login ke Hugging Face menggunakan Secret Colab!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets evaluate sentencepiece accelerate rouge-score -q

import shutil

# ======================================================
# folder tempat data latih csv
DRIVE_DATA_DIR = '/content/drive/MyDrive/data_latih'
# ======================================================

shutil.copy(f'{DRIVE_DATA_DIR}/train.csv', 'train.csv')
shutil.copy(f'{DRIVE_DATA_DIR}/test.csv',  'test.csv')
print("Data berhasil di-copy dari Google Drive!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
Data berhasil di-copy dari Google Drive!


## 3. Import & Konfigurasi

In [3]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
import pandas as pd
import evaluate
import torch
import shutil
import itertools
from pathlib import Path

MODEL_NAME = "Wikidepia/IndoT5-base"
DATA_DIR    = Path('.')
OUTPUT_ROOT = Path('/content/drive/MyDrive/models')
BEST_DIR    = OUTPUT_ROOT / 'indot5_finetuned'
LOG_DIR     = OUTPUT_ROOT / 'training_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

PREFIX  = 'ringkas: '
MAX_IN  = 512
MAX_OUT = 512

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')

Device : cuda
Model  : Wikidepia/IndoT5-base


## 4. Data Loading & Tokenization

In [5]:
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_test  = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train: {len(df_train)} | Test: {len(df_test)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenisasi(examples):
    inputs = [PREFIX + s for s in examples['source']]
    m_in   = tokenizer(inputs, max_length=MAX_IN, truncation=True, padding='max_length')
    labels = tokenizer(
        text_target=examples['target'],
        max_length=MAX_OUT, truncation=True, padding='max_length'
    )
    m_in['labels'] = [
        [-100 if t == tokenizer.pad_token_id else t for t in seq]
        for seq in labels['input_ids']
    ]
    return m_in

train_ds = Dataset.from_pandas(df_train).map(tokenisasi, batched=True, remove_columns=df_train.columns.tolist())
test_ds  = Dataset.from_pandas(df_test).map(tokenisasi, batched=True, remove_columns=df_test.columns.tolist())
print('Tokenisasi selesai.')

Train: 335 | Test: 84


config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/777k [00:00<?, ?B/s]

Map:   0%|          | 0/335 [00:00<?, ? examples/s]

Map:   0%|          | 0/84 [00:00<?, ? examples/s]

Tokenisasi selesai.


## 5. Diagnostik

In [6]:
# log Cek apa yang dihasilkan model saat ini
model_test = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
tokenizer_test = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_input = PREFIX + df_test['source'].iloc[0][:300]
inputs = tokenizer_test(sample_input, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
outputs = model_test.generate(**inputs, max_new_tokens=100)
print("PREFIX yang dipakai:", repr(PREFIX))
print("Input awal:", sample_input[:100])
print("Output model:", tokenizer_test.decode(outputs[0], skip_special_tokens=True))
print("Referensi:", df_test['target'].iloc[0][:200])

# Cleanup setelah diagnostik
del model_test, tokenizer_test
import torch, gc
torch.cuda.empty_cache()
gc.collect()
print("VRAM diagnostik sudah dibersihkan.")



pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

PREFIX yang dipakai: 'ringkas: '
Input awal: ringkas: untuk disahkan menjadi undang-undang setuju Terima kasih selanjutnya kami bersilapkan kepad
Output model: Terima kasih atas perhatiannya kami bersilakan kepada yang terhormat so dari Terima kasih Menteri Pemberian Perempuan dan Perinduhan Anak Terima kasih atas perhatiannya kami bersilakan Terima kasih atas perhatiannya kami ber Terima kasih selanjutnya,-undang-undang dan yang terhormat anak anak yang ter terhorm
Referensi: organisasi dan elemen masyarakat yang terlibat dalam pembahasan, tentunya secara khusus kami sampaikan terima kasih sebesar besarnya kepada pemerhati isu anak, pembela hak anak, pendamping hak anak, m
VRAM diagnostik sudah dibersihkan.


## 6. Fungsi Inti Trainer & ROUGE

In [7]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
import pandas as pd
import evaluate
import torch
import shutil
import itertools
from pathlib import Path

MODEL_NAME = "Wikidepia/IndoT5-base"
DATA_DIR    = Path('.')
OUTPUT_ROOT = Path('/content/drive/MyDrive/models')
BEST_DIR    = OUTPUT_ROOT / 'indot5_finetuned'
LOG_DIR     = OUTPUT_ROOT / 'training_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

PREFIX  = 'ringkas: '
MAX_IN  = 512
MAX_OUT = 512

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')

rouge = evaluate.load('rouge')

def hitung_rouge(model, test_dataset):
    """Hitung ROUGE dengan model yang sudah ditraining"""
    from torch.utils.data import DataLoader

    # Set model ke mode evaluasi
    model.eval()
    preds = []
    refs  = df_test['target'].tolist()

    test_dataset.set_format('torch')
    loader = DataLoader(test_dataset, batch_size=2)

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)

            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_OUT,
                no_repeat_ngram_size=3,
                # repetition_penalty=2.0,
                num_beams=4,
                early_stopping=True,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            preds.extend(decoded)

    # Bersihkan prefix dari prediksi
    preds = [p.replace(PREFIX, '').strip() for p in preds]

    if preds:
        print(f"  [Cek Prediksi] Model : {preds[0][:100]}...")
        print(f"  [Cek Referensi] Asli: {refs[0][:100]}...")

    pairs = [(p, r) for p, r in zip(preds, refs) if p.strip()]

    if not pairs:
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    p_clean, r_clean = zip(*pairs)
    scores = rouge.compute(predictions=list(p_clean), references=list(r_clean), use_stemmer=False)
    return {k: round(v, 4) for k, v in scores.items()}

def buat_trainer(model, params, run_dir):
    """Buat Seq2SeqTrainer VERSI SULTAN (A100 GPU) - Ngebut Maksimal!"""

    target_bs = params['per_device_train_batch_size']

    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=params['num_train_epochs'],

        per_device_train_batch_size=target_bs,
        gradient_accumulation_steps=1,

        per_device_eval_batch_size=4,
        learning_rate=params['learning_rate'],
        warmup_steps=50,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,

        save_total_limit=1,

        predict_with_generate=True,
        generation_max_length=MAX_OUT,

        # GPU A100
        fp16=False,
        bf16=True,

        report_to='none',
    )

    return Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=test_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
        processing_class=tokenizer,
    )

def jalankan_satu_run(run_label, params, run_dir, hasil_list, best_state):
    """Training + evaluasi 1 run, simpan HANYA model global terbaik, lalu bersihkan sampah."""
    print(f'\n{"="*55}')
    print(f'[{run_label}] Epoch={params["num_train_epochs"]}  '
          f'BS={params["per_device_train_batch_size"]}  '
          f'LR={params["learning_rate"]}')
    print(f'{"="*55}')

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    trainer = buat_trainer(model, params, run_dir)

    # Mulai Training
    trainer.train()

    # Ambil model yang sudah matang dari dalam trainer
    model = trainer.model

    # Hitung ROUGE
    scores = hitung_rouge(model, test_ds)
    print(f'  ROUGE-1={scores["rouge1"]}  ROUGE-2={scores["rouge2"]}  ROUGE-L={scores["rougeL"]}')

    hasil_list.append({
        'run'          : run_label,
        'epoch'        : params['num_train_epochs'],
        'batch_size'   : params['per_device_train_batch_size'],
        'learning_rate': params['learning_rate'],
        'rouge1'       : scores['rouge1'],
        'rouge2'       : scores['rouge2'],
        'rougeL'       : scores['rougeL'],
    })

    # Cek Apakah Model Terbaik?
    if scores['rougeL'] > best_state['rouge_l']:
        best_state['rouge_l'] = scores['rougeL']
        best_state['config']  = params.copy()

        # JIKA IYA: Save model terbaik ke folder BEST_DIR (akan otomatis menimpa model lama)
        trainer.save_model(str(BEST_DIR))
        tokenizer.save_pretrained(str(BEST_DIR))
        print(f'⭐ Model terbaik baru! ROUGE-L={scores["rougeL"]} (Tersimpan di {BEST_DIR})')

    # BERSIH-BERSIH (HAPUS FOLDER EKSPERIMEN INI DARI DRIVE)
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if run_dir.exists():
        shutil.rmtree(run_dir)
        print(f'Folder {run_dir.name} dihapus (Hemat Penyimpanan!).')

Device : cuda
Model  : Wikidepia/IndoT5-base


## 6. Eksekusi 19 Hyperparameter

In [ ]:
import itertools
import pandas as pd
from transformers import AutoModelForSeq2SeqLM
import torch

hasil_eksperimen = []
best_state = {'rouge_l': -1, 'config': None, 'dir': None}

# =====================================================================
# TAHAP 1: PENGUJIAN BASELINE (ZERO-SHOT)
# =====================================================================
print("MENGUJI EXP-BASE: ZERO-SHOT (MODEL MENTAHAN TANPA TRAINING)")

# Load model mentah dari HuggingFace
model_mentah = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Langsung meringkas data test (Zero-Shot)
skor_baseline = hitung_rouge(model_mentah, test_ds)
print(f"  [EXP-BASE] ROUGE-1: {skor_baseline['rouge1']} | ROUGE-2: {skor_baseline['rouge2']} | ROUGE-L: {skor_baseline['rougeL']}")

hasil_eksperimen.append({
    'run'          : 'EXP-BASE (Zero-Shot)',
    'epoch'        : 0,
    'batch_size'   : '-',
    'learning_rate': '-',
    'rouge1'       : skor_baseline['rouge1'],
    'rouge2'       : skor_baseline['rouge2'],
    'rougeL'       : skor_baseline['rougeL'],
})

# Bersihkan VRAM GPU
del model_mentah
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =====================================================================
# TAHAP 2: PENGUJIAN HYPERPARAMETER (FINE-TUNING)
# =====================================================================
epochs = [5, 10, 15]
batch_sizes = [4, 8]
learning_rates = [1e-4, 5e-5, 3e-5]

kombinasi = list(itertools.product(epochs, batch_sizes, learning_rates))
print(f"\nMEMULAI {len(kombinasi)} EKSPERIMEN FINE-TUNING...")

for i, (ep, bs, lr) in enumerate(kombinasi, 1):
    run_label = f"EXP-{i:03d}"
    params = {
        'num_train_epochs': ep,
        'per_device_train_batch_size': bs,
        'learning_rate': lr
    }

    run_dir = OUTPUT_ROOT / f"run_{run_label}_ep{ep}_bs{bs}_lr{lr}"
    jalankan_satu_run(run_label, params, run_dir, hasil_eksperimen, best_state)

# =====================================================================
# TAHAP 3: TAMPILKAN HASIL AKHIR
# =====================================================================
df_hasil = pd.DataFrame(hasil_eksperimen)
print("\nREKAPITULASI HASIL EKSPERIMEN")
display(df_hasil)

df_hasil.to_csv(OUTPUT_ROOT / 'rekap_eksperimen_skripsi.csv', index=False)
print("File CSV rekapitulasi berhasil disimpan di Drive!")

## 8. Visualisasi Hasil

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Urutkan berdasarkan ROUGE-L tertinggi
df_hasil_sorted = df_hasil.sort_values('rougeL', ascending=False).reset_index(drop=True)

# Grafik Keseluruhan
fig, ax = plt.subplots(figsize=(14, 4))
# Warnai merah khusus untuk Zero-Shot
colors = ['#E74C3C' if 'Zero-Shot' in r else '#4A90D9' for r in df_hasil_sorted['run']]
ax.bar(df_hasil_sorted['run'], df_hasil_sorted['rougeL'], color=colors)

ax.set_title('ROUGE-L per Eksperimen (Merah = Zero-Shot Baseline)', fontsize=12)
ax.set_xlabel('Skenario / Run')
ax.set_ylabel('Skor ROUGE-L')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'rouge_per_run.png', dpi=150)
plt.show()

# Grafik Pengaruh Hyperparameter (Hanya dari hasil Fine-Tuning)
df_grid = df_hasil[df_hasil['run'] != 'EXP-BASE (Zero-Shot)']

fig2, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
fig2.suptitle('Pengaruh Hyperparameter terhadap ROUGE-L', fontsize=12)

for ax, col, label in zip(axes, ['epoch', 'batch_size', 'learning_rate'], ['Epoch', 'Batch Size', 'Learning Rate']):
    g = df_grid.groupby(col)['rougeL'].mean().reset_index()
    ax.bar(g[col].astype(str), g['rougeL'], color='#2ECC71')
    ax.set_title(f'ROUGE-L vs {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('Rata-rata ROUGE-L')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'visualisasi_hyperparameter.png', dpi=150)
plt.show()
print('Grafik siap!')